# 2.2 注意力机制优化

## 什么是注意力优化？

标准自注意力的计算复杂度为$O(n^2)$，需要存储$n \times n$的注意力矩阵，是长序列推理的核心瓶颈。注意力优化通过改进计算方式、减少KV数量或跳过不重要计算来降低延迟和内存。

## 为什么注意力是瓶颈？

标准注意力的计算过程：
$$\text{Attn}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

其中：
- $Q \in \mathbb{R}^{n \times d_k}$：查询矩阵
- $K \in \mathbb{R}^{n \times d_k}$：键矩阵
- $V \in \mathbb{R}^{n \times d_v}$：值矩阵
- $QK^T \in \mathbb{R}^{n \times n}$：注意力分数矩阵，这是$O(n^2)$的根源
- $d_k$：每个头的维度

当$n=4096$时，注意力矩阵需要$4096 \times 4096 \times 4 = 64$MB（FP32），且随序列长度平方增长。

## 注意力优化的三大方向

1. **高效计算**：Flash Attention等，不改变数学结果，优化硬件执行
2. **架构优化**：MQA/GQA等，减少KV数量，降低KV Cache和计算量
3. **稀疏计算**：跳过不重要的注意力对，降低计算复杂度

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch version: {torch.__version__}")

---
## 2.2.1 高效注意力计算

### Flash Attention 原理演示

#### 什么是Flash Attention？

Flash Attention通过分块计算（tiling）和重计算（recomputation）策略，减少对HBM（高带宽内存）的读写次数，将注意力计算变为IO-bound而非compute-bound。不改变数学结果，是精确注意力的硬件高效实现。

#### 为什么Flash Attention有效？

1. **内存墙问题**：标准注意力需要将$O(n^2)$的注意力矩阵写入HBM再读回，内存带宽成为瓶颈。Flash Attention将计算分块在SRAM中完成，避免存储完整注意力矩阵。
2. **减少HBM访问**：标准注意力需要$O(n^2)$次HBM读写，Flash Attention仅需$O(n^2 d^2 / M)$次，其中$M$是SRAM大小。
3. **数学精确等价**：通过在线softmax算法（online softmax），分块计算的结果与标准计算完全一致。

#### Flash Attention的数学原理

核心是在线softmax（Online Softmax），分块计算时维护运行统计量：

$$m^{(j)} = \max(m^{(j-1)}, \max(S^{(j)}))$$
$$\ell^{(j)} = e^{m^{(j-1)} - m^{(j)}} \ell^{(j-1)} + \sum_c e^{S^{(j)}_c - m^{(j)}}$$
$$O^{(j)} = \text{diag}(e^{m^{(j-1)} - m^{(j)}}) O^{(j-1)} + \text{softmax}(S^{(j)}) V^{(j)}$$

其中：
- $S^{(j)} = Q K^{(j)T} / \sqrt{d_k}$：第$j$块的注意力分数
- $m^{(j)}$：到第$j$块为止的行最大值
- $\ell^{(j)}$：到第$j$块为止的softmax归一化因子
- $O^{(j)}$：到第$j$块为止的累积输出
- $e^{m^{(j-1)} - m^{(j)}}$：修正因子，确保之前块的结果在新的最大值下正确归一化

In [ ]:
def standard_attention(Q, K, V):
    """标准注意力实现 - O(N²)内存"""
    head_dim = Q.shape[-1]
    attn_weights = torch.matmul(Q, K.transpose(-2, -1)) / (head_dim ** 0.5)
    attn_probs = F.softmax(attn_weights, dim=-1)
    output = torch.matmul(attn_probs, V)
    return output, attn_weights

def flash_attention_tiled(Q, K, V, block_size=64):
    """Flash Attention分块计算原理演示
    分块计算softmax，避免存储完整N×N注意力矩阵"""
    B, H, N, D = Q.shape
    output = torch.zeros_like(Q)
    row_max = torch.full((B, H, N, 1), float('-inf'), device=Q.device)
    row_sum = torch.zeros((B, H, N, 1), device=Q.device)

    for j in range(0, N, block_size):
        K_block = K[:, :, j:j+block_size, :]
        V_block = V[:, :, j:j+block_size, :]
        S_block = torch.matmul(Q, K_block.transpose(-2, -1)) / (D ** 0.5)

        block_max = S_block.max(dim=-1, keepdim=True).values
        new_max = torch.maximum(row_max, block_max)
        exp_correction = torch.exp(row_max - new_max)
        block_exp = torch.exp(S_block - new_max)

        row_sum = row_sum * exp_correction + block_exp.sum(dim=-1, keepdim=True)
        output = output * exp_correction + torch.matmul(block_exp, V_block)
        row_max = new_max

    output = output / row_sum
    return output

B, H, N, D = 1, 4, 256, 64
Q = torch.randn(B, H, N, D)
K = torch.randn(B, H, N, D)
V = torch.randn(B, H, N, D)

out_standard, attn_weights = standard_attention(Q, K, V)
out_flash = flash_attention_tiled(Q, K, V, block_size=64)

diff = (out_standard - out_flash).norm() / out_standard.norm() * 100

standard_mem = B * H * N * N * 4
flash_mem = B * H * N * D * 4 * 3

print(f"=== Flash Attention vs 标准注意力 ===")
print(f"序列长度: {N}, 头数: {H}, 头维度: {D}")
print(f"输出差异: {diff:.6f}% (数学等价)")
print(f"\n内存占用:")
print(f"  标准注意力(N×N矩阵): {standard_mem/1024:.1f} KB")
print(f"  Flash Attention(无N×N): {flash_mem/1024:.1f} KB")
print(f"  节省: {(1-flash_mem/standard_mem)*100:.1f}%")

for seq_len in [256, 512, 1024, 2048, 4096, 8192]:
    std_mem = B * H * seq_len * seq_len * 4 / 1024 / 1024
    fl_mem = B * H * seq_len * D * 4 * 3 / 1024 / 1024
    print(f"  N={seq_len:<6} 标准={std_mem:.1f}MB Flash={fl_mem:.1f}MB 节省{(1-fl_mem/std_mem)*100:.0f}%")

---
## 2.2.2 注意力架构优化

### MQA / GQA 原理与实现

#### 什么是MQA和GQA？

- **MHA (Multi-Head Attention)**：每个头有独立的Q/K/V投影，$h$个头有$h$组KV
- **MQA (Multi-Query Attention)**：所有头共享K/V投影，仅Q保持多头，KV只有1组
- **GQA (Grouped-Query Attention)**：Q头分组，每组共享K/V，在MHA和MQA间取折中

#### 为什么MQA/GQA有效？

1. **KV Cache大幅减少**：推理时KV Cache与KV头数成正比。MQA将KV头数从$h$降到1，KV Cache减少$h$倍；GQA降到$g$组，减少$h/g$倍。
2. **推理速度提升**：KV Cache减少意味着更少的内存搬运，自回归生成速度显著提升。
3. **精度损失可控**：研究表明GQA（如8组KV头）的精度接近MHA，而MQA在某些任务上可能有轻微下降。

#### MQA/GQA的数学公式

**MHA**：$h$个头各自计算
$$\text{head}_i = \text{Attn}(QW_i^Q, KW_i^K, VW_i^V), \quad i = 1, ..., h$$

**MQA**：所有头共享KV
$$\text{head}_i = \text{Attn}(QW_i^Q, KW^K, VW^V), \quad i = 1, ..., h$$

**GQA**：$g$组共享KV，每组$h/g$个Q头
$$\text{head}_i = \text{Attn}(QW_i^Q, KW_{\lfloor i/(h/g) \rfloor}^K, VW_{\lfloor i/(h/g) \rfloor}^V)$$

其中：
- $h$：Q头数（总头数）
- $g$：KV头组数（GQA），MQA时$g=1$，MHA时$g=h$
- $W_i^Q \in \mathbb{R}^{d \times d_k}$：第$i$个Q头的投影矩阵
- $W_j^K, W_j^V$：第$j$组KV的投影矩阵
- KV Cache大小：$2 \times g \times L \times d_k$（与Q头数无关）

In [ ]:
class MultiHeadAttention(nn.Module):
    """标准多头注意力 (MHA)"""
    def __init__(self, dim, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.out_proj(out), k, v

class MultiQueryAttention(nn.Module):
    """多查询注意力 (MQA): 所有头共享K/V"""
    def __init__(self, dim, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, self.head_dim)
        self.v_proj = nn.Linear(dim, self.head_dim)
        self.out_proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, 1, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, 1, self.head_dim).transpose(1, 2)
        k = k.expand(-1, self.n_heads, -1, -1)
        v = v.expand(-1, self.n_heads, -1, -1)
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.out_proj(out), k, v

class GroupedQueryAttention(nn.Module):
    """分组查询注意力 (GQA): Q头分组，每组共享K/V"""
    def __init__(self, dim, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = dim // n_heads
        self.n_rep = n_heads // n_kv_heads
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(dim, n_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.n_kv_heads, self.head_dim).transpose(1, 2)
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.out_proj(out), k, v

dim, n_heads = 512, 8
seq_len = 128
x = torch.randn(1, seq_len, dim)

mha = MultiHeadAttention(dim, n_heads)
mqa = MultiQueryAttention(dim, n_heads)
gqa = GroupedQueryAttention(dim, n_heads, n_kv_heads=2)

with torch.no_grad():
    out_mha, k_mha, v_mha = mha(x)
    out_mqa, k_mqa, v_mqa = mqa(x)
    out_gqa, k_gqa, v_gqa = gqa(x)

mha_kv_params = dim * n_heads * 2
mqa_kv_params = dim * (dim // n_heads) * 2
gqa_kv_params = dim * (2 * (dim // n_heads)) * 2

mha_kv_cache = 1 * n_heads * seq_len * (dim // n_heads) * 2 * 2
mqa_kv_cache = 1 * 1 * seq_len * (dim // n_heads) * 2 * 2
gqa_kv_cache = 1 * 2 * seq_len * (dim // n_heads) * 2 * 2

print(f"=== MHA vs MQA vs GQA 对比 ===")
print(f"dim={dim}, n_heads={n_heads}, GQA n_kv_heads=2")
print(f"\n{'方法':<8} {'KV参数量':<15} {'KV Cache/seq_len':<20} {'KV Cache节省':<15}")
print("-" * 58)
print(f"{'MHA':<8} {mha_kv_params:<15,} {mha_kv_cache/1024:.1f} KB/tok   {'基准':<15}")
print(f"{'MQA':<8} {mqa_kv_params:<15,} {mqa_kv_cache/1024:.1f} KB/tok   {(1-mqa_kv_cache/mha_kv_cache)*100:.0f}%")
print(f"{'GQA':<8} {gqa_kv_params:<15,} {gqa_kv_cache/1024:.1f} KB/tok   {(1-gqa_kv_cache/mha_kv_cache)*100:.0f}%")
print(f"\nMQA: KV Cache减少到1/{n_heads}，但可能影响精度")
print(f"GQA: KV Cache减少到{n_heads//2}/{n_heads}，精度与效率的平衡")

---
## 2.2.3 稀疏注意力

### 什么是稀疏注意力？

通过选择性地计算部分注意力对，跳过大部分注意力计算，将复杂度从$O(n^2)$降低。稀疏注意力基于一个关键观察：注意力矩阵通常是稀疏的，大部分注意力权重集中在少数位置。

### 为什么稀疏注意力有效？

1. **注意力稀疏性**：研究表明LLM的注意力分布高度稀疏，少数token获得大部分注意力权重，大部分注意力对可以安全跳过。
2. **局部性**：自然语言中相邻token的依赖关系最强，局部窗口内的注意力最重要。
3. **全局信息**：少数全局token（如[CLS]、句首）可以传递全局信息，弥补局部注意力的不足。

### 局部注意力 + 全局注意力混合

Longformer等模型采用的策略：大部分token只计算局部窗口内的注意力，少数全局token与所有位置计算注意力。

计算复杂度：$O(n \cdot w + n \cdot g)$，其中$w$为窗口大小，$g$为全局token数，远小于$O(n^2)$。

In [ ]:
class SparseAttention(nn.Module):
    """稀疏注意力：局部窗口 + 全局token
    注意：本实现为教学演示，仍计算完整注意力矩阵后应用mask，
    不减少实际FLOPs。实际稀疏注意力应跳过mask为0的注意力对计算，
    需要自定义CUDA kernel或使用Block Sparse Attention等库。"""
    def __init__(self, dim, n_heads, window_size=64, n_global_tokens=1):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.window_size = window_size
        self.n_global_tokens = n_global_tokens
        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)

    def _create_sparse_mask(self, seq_len):
        """创建稀疏注意力mask"""
        mask = torch.zeros(seq_len, seq_len, dtype=torch.bool)
        for i in range(seq_len):
            start = max(0, i - self.window_size // 2)
            end = min(seq_len, i + self.window_size // 2 + 1)
            mask[i, start:end] = True
            mask[i, :self.n_global_tokens] = True
            mask[:self.n_global_tokens, i] = True
        return mask

    def forward(self, x):
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)

        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        sparse_mask = self._create_sparse_mask(N).unsqueeze(0).unsqueeze(0)
        attn = attn.masked_fill(~sparse_mask, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        attn = attn.masked_fill(attn.isnan(), 0)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.out_proj(out)

dim, n_heads = 256, 4
window_sizes = [32, 64, 128]

print(f"=== 稀疏注意力计算量对比 ===")
print(f"\n{'方法':<30} {'注意力对数':<15} {'占比':<10}")
print("-" * 55)

for seq_len in [256, 512, 1024]:
    full_pairs = seq_len * seq_len
    print(f"\n序列长度 = {seq_len}")
    print(f"  {'全注意力':<28} {full_pairs:<15,} {'100%':<10}")
    for ws in window_sizes:
        local_pairs = seq_len * ws
        global_pairs = seq_len * 1 + 1 * seq_len
        total_pairs = local_pairs + global_pairs
        ratio = total_pairs / full_pairs * 100
        print(f"  {'局部(ws='+str(ws)+')+全局':<28} {total_pairs:<15,} {ratio:<10.1f}%")

sparse_attn = SparseAttention(dim, n_heads, window_size=64, n_global_tokens=1)
x = torch.randn(1, 128, dim)
out = sparse_attn(x)
print(f"\n稀疏注意力输出形状: {out.shape}")

### 注意力优化方法综合对比

不同注意力优化方法在计算复杂度、精度影响和硬件需求方面的对比。选择方法时需根据部署场景和硬件条件决定。

In [ ]:
print(f"{'方法':<25} {'复杂度':<15} {'精度影响':<15} {'硬件需求':<15}")
print("-" * 70)
methods = [
    ("Flash Attention", "O(N²)计算", "无(精确)", "GPU SRAM"),
    ("MQA", "O(N²)计算", "较小", "通用"),
    ("GQA", "O(N²)计算", "很小", "通用"),
    ("局部注意力", "O(N·W)", "丢失远距", "通用"),
    ("局部+全局", "O(N·W+N)", "较小", "通用"),
    ("稀疏模式", "O(N·√N)", "中等", "通用"),
]
for name, comp, impact, hw in methods:
    print(f"{name:<25} {comp:<15} {impact:<15} {hw:<15}")

print(f"\n=== 产业级选择建议 ===")
print(f"1. GPU端侧: Flash Attention + GQA (精度与效率最优平衡)")
print(f"2. CPU/NPU端侧: GQA + 局部注意力 (减少KV Cache和计算量)")
print(f"3. 长序列: 局部+全局稀疏注意力 (线性复杂度)")
print(f"4. 极致速度: MQA (最小KV Cache，适合短序列)")

### 产业级实战：PyTorch原生高效注意力 + 真实性能对比

PyTorch 2.0+ 提供了 `F.scaled_dot_product_attention` (SDPA)，它自动选择最优后端：
- **Flash Attention** (CUDA, 计算密集型)
- **Memory-Efficient Attention** (xFormers风格, 内存密集型)
- **Math Implementation** (回退实现)

这是产业界最常用的注意力优化方式——无需安装额外库，一行代码即可获得Flash Attention级别的性能。

In [ ]:
import time

def manual_attention(q, k, v, mask=None):
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn = F.softmax(scores, dim=-1)
    return torch.matmul(attn, v)

def sdpa_attention(q, k, v, mask=None):
    attn_mask = mask.to(dtype=q.dtype) if mask is not None else None
    return F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)

configs = [
    (4, 512, 64),
    (4, 1024, 64),
    (4, 2048, 64),
]

print(f"=== PyTorch SDPA vs 手动注意力 性能对比 ===")
print(f"{'Batch×Heads':<14} {'SeqLen':<8} {'手动(ms)':<12} {'SDPA(ms)':<12} {'加速比':<10}")
print("-" * 56)

for n_heads, seq_len, head_dim in configs:
    q = torch.randn(1, n_heads, seq_len, head_dim)
    k = torch.randn(1, n_heads, seq_len, head_dim)
    v = torch.randn(1, n_heads, seq_len, head_dim)

    with torch.no_grad():
        for _ in range(5):
            manual_attention(q, k, v)
        start = time.perf_counter()
        for _ in range(20):
            manual_attention(q, k, v)
        manual_time = (time.perf_counter() - start) / 20 * 1000

        for _ in range(5):
            sdpa_attention(q, k, v)
        start = time.perf_counter()
        for _ in range(20):
            sdpa_attention(q, k, v)
        sdpa_time = (time.perf_counter() - start) / 20 * 1000

    speedup = manual_time / sdpa_time
    print(f"{n_heads:<14} {seq_len:<8} {manual_time:<12.2f} {sdpa_time:<12.2f} {speedup:<10.2f}x")

with torch.no_grad():
    q = torch.randn(1, 4, 64, 64)
    k = torch.randn(1, 4, 64, 64)
    v = torch.randn(1, 4, 64, 64)
    out_manual = manual_attention(q, k, v)
    out_sdpa = sdpa_attention(q, k, v)
    max_diff = (out_manual - out_sdpa).abs().max().item()

print(f"\n数值一致性: 最大差异 = {max_diff:.8f} ({'PASS' if max_diff < 1e-5 else 'FAIL'})")

with torch.backends.cuda.sdp_kernel(enable_flash=True, enable_mem_efficient=True, enable_math=True):
    backend_info = "Flash Attention + Mem Efficient + Math 均可用"
print(f"\n可用后端: {backend_info if torch.cuda.is_available() else 'CPU模式 (Math实现)'}")

print(f"\n产业界注意力优化最佳实践:")
print(f"1. 首选: F.scaled_dot_product_attention (自动选最优后端)")
print(f"2. GPU: 自动使用Flash Attention (2-4x加速)")
print(f"3. CPU: 使用Math实现 (与手动等价)")
print(f"4. 长序列: 配合GQA减少KV → 再用SDPA加速")
print(f"5. 自定义: torch.backends.cuda.sdp_kernel 控制后端选择")